# Model Comparison with considerations for Edge Deployment

This notebook compares lightweight CNN architectures to determine the best model for 
driver‑behavior classification under edge‑computing constraints.

Architectures evaluated:
- MobileNetV3‑Small
- EfficientNet‑B0
- ShuffleNetV2
- ResNet18 (baseline)


In [1]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
import time
import pandas as pd

from pathlib import Path
import sys

sys.path.append("..")
from dataset import DriverDataset, get_transforms



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\viper\Documents\Coding\driver-behaviour-classifier\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\viper\Documents\Coding\driver-behaviour-classifier\.venv\Lib\site-packages\traitlets\config\application.py", line 1082, in launch_instance
    app.start()
  File "c:\Users\viper\Documents\Coding\driver-behaviour-classifier\.venv\Lib\site-p

## Load Dataset

In [2]:
train_tf, val_tf = get_transforms()

train_ds = DriverDataset("../data/imgs/train", transform=train_tf)
val_ds   = DriverDataset("../data/imgs/train", transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)


## Define Models

In [3]:
def build_mobilenet():
    m = models.mobilenet_v3_small(weights="IMAGENET1K_V1")
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, 10)
    return m

def build_efficientnet():
    m = models.efficientnet_b0(weights="IMAGENET1K_V1")
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 10)
    return m

def build_shufflenet():
    m = models.shufflenet_v2_x1_0(weights="IMAGENET1K_V1")
    m.fc = nn.Linear(m.fc.in_features, 10)
    return m

def build_resnet18():
    m = models.resnet18(weights="IMAGENET1K_V1")
    m.fc = nn.Linear(m.fc.in_features, 10)
    return m


## Compare Model Complexity

In [4]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

models_to_test = {
    "MobileNetV3-Small": build_mobilenet(),
    "EfficientNet-B0": build_efficientnet(),
    "ShuffleNetV2": build_shufflenet(),
    "ResNet18": build_resnet18()
}

for name, model in models_to_test.items():
    print(name, "params:", count_params(model))


MobileNetV3-Small params: 1528106
EfficientNet-B0 params: 4020358
ShuffleNetV2 params: 1263854
ResNet18 params: 11181642


## Inference Latency Benchmark

In [5]:
def benchmark(model, device="cpu", runs=50):
    model.eval().to(device)
    dummy = torch.randn(1, 3, 224, 224).to(device)

    # warmup
    for _ in range(10):
        _ = model(dummy)

    start = time.time()
    for _ in range(runs):
        _ = model(dummy)
    end = time.time()

    return (end - start) / runs

latencies = {}
for name, model in models_to_test.items():
    latencies[name] = benchmark(model)

latencies


{'MobileNetV3-Small': 0.012094621658325195,
 'EfficientNet-B0': 0.022146077156066896,
 'ShuffleNetV2': 0.013005671501159667,
 'ResNet18': 0.010816702842712403}

Latency somewhat aligns with model complexity, all are similar except EfficientNet-B0 being more complex and slightly slower than others.

## Quick Training Runs

In [6]:
def quick_train(model, epochs=2):
    model = model.to("cuda" if torch.cuda.is_available() else "cpu")
    device = next(model.parameters()).device

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            opt.zero_grad()
            out = model(imgs)
            loss = loss_fn(out, labels)
            loss.backward()
            opt.step()

    # quick validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            preds = out.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total


In [7]:
results = []

for name, model in models_to_test.items():
    acc = quick_train(model, epochs=2)
    results.append({
        "model": name,
        "params": count_params(model),
        "latency_ms": latencies[name] * 1000,
        "val_acc_2_epochs": acc
    })

pd.DataFrame(results)


RuntimeError: Numpy is not available

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch

ds = TensorDataset(torch.randn(10,3,224,224), torch.zeros(10, dtype=torch.long))

for nw in [0, 1]:
    print("Testing num_workers =", nw)
    try:
        dl = DataLoader(ds, batch_size=2, num_workers=nw)
        for x, y in dl:
            pass
        print("  OK")
    except Exception as e:
        print("  FAILED:", e)